# 🚀 Phase 3: QLoRA Training Loop — Production Hardened

This notebook contains the complete production-hardened training pipeline. It loads the **Qwen2.5-3B-Instruct** base model, quantizes it to 4-bit using `bitsandbytes`, configures a PEFT LoRA adapter with creation-time type safety, formats ChatML datasets, and executes fine-tuning using modern `trl` SFTTrainer.

**Target environment:** `transformers>=4.45.0` | `trl>=0.11.0` | `peft>=0.12.0` | `accelerate>=0.34.0` | `bitsandbytes>=0.43.0` | `torch>=2.0.0`

## 1. Define PROJECT_ROOT and Verify Workspace Directories
We mount Google Drive (if on Google Colab) and verify that all necessary folders exist under our `PROJECT_ROOT`.

In [ ]:
from pathlib import Path
import sys

# Determine if running on Google Colab
IN_COLAB = 'google.colab' in sys.modules

# Define PROJECT_ROOT exactly once
if 'PROJECT_ROOT' not in globals():
    PROJECT_ROOT = Path("/content/drive/MyDrive/LLM-Studio") if IN_COLAB else Path("../../").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

if IN_COLAB:
    print("Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')

print(f"PROJECT_ROOT resolved to: {PROJECT_ROOT}")

# Verify required directories
required_dirs = [
    PROJECT_ROOT / "training",
    PROJECT_ROOT / "training/configs",
    PROJECT_ROOT / "training/scripts",
    PROJECT_ROOT / "data/raw",
    PROJECT_ROOT / "data/processed",
    PROJECT_ROOT / "models",
    PROJECT_ROOT / "models/adapters",
    PROJECT_ROOT / "models/merged"
]

print("\nAuditing required project directories...")
for folder in required_dirs:
    if not folder.exists():
        raise FileNotFoundError(
            f"Missing required project directory: {folder.relative_to(PROJECT_ROOT) if PROJECT_ROOT in folder.parents else folder}\n"
            f"Please ensure the project is correctly set up under PROJECT_ROOT: {PROJECT_ROOT}"
        )
    print(f"  ✓ {folder.relative_to(PROJECT_ROOT) if PROJECT_ROOT in folder.parents else folder} is present.")
print("✓ Project structure verification complete.")

## 2. Library Version Audit & System Capability Diagnostics
Verify the installed library versions and inspect GPU compute capability via central utilities.

In [ ]:
from training.scripts import utils
import sys

print("--- Dependency Validation ---")
warnings = utils.validate_dependencies()
for w in warnings:
    print(w)

gpu_info = utils.get_gpu_info()
utils.log_system_info({}, gpu_info)

## 3. Load and Validate Configuration
We load the YAML config file directly using our verified `PROJECT_ROOT` paths and run schema validation.

In [ ]:
import yaml
import json
from training.scripts import utils

CONFIG_DIR = PROJECT_ROOT / "training/configs"
config_path = CONFIG_DIR / "qlora_config.yaml"

if not config_path.exists():
    raise FileNotFoundError(f"Configuration file missing at: {config_path}")

print(f"Loading config from: {config_path}")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# Enforce schema validation
utils.validate_config_schema(config, PROJECT_ROOT)

model_cfg = config["model"]
train_cfg = config["training"]
lora_cfg = config["lora"]
dataset_cfg = config["dataset"]

print("✓ Config loaded and schema validated successfully via central utilities!")

## 4. Automatic Precision & Hardware Auto-Detection
Derive FP16 vs BF16 compute dtypes cleanly from hardware capabilities using central utility solvers.

In [ ]:
from training.scripts import utils

gpu_info = utils.get_gpu_info()
fp16_mode, bf16_mode, compute_dtype, precision_desc = utils.resolve_precision_mode(train_cfg, gpu_info)

print(f"--- Precision Resolution ---")
print(f"Selected Mode: {precision_desc}")
print(f"Compute Dtype: {compute_dtype}")
print(f"fp16 flag: {train_cfg.get('fp16')} | bf16 flag: {train_cfg.get('bf16')}")
print(f"Hardware BF16 capability detected: {gpu_info['devices'][0]['bf16_supported'] if gpu_info['cuda_available'] else False} (not used as override)")

## 5. Initialize Experiment Folder & Global Seed
Initialize random seeds and create distinct tracking directories under `artifacts/experiments/` to store execution metadata.

In [ ]:
from datetime import datetime, timezone
import yaml
import json
from training.scripts import utils

# Initialize global seeds across random, numpy, torch, transformers
utils.set_global_seed(dataset_cfg.get("random_seed", 42))
print("✓ Global random seed initialized.")

timestamp = datetime.now().strftime("%Y_%m_%d_%H%M%S")
exp_dir_path = PROJECT_ROOT / train_cfg["experiments_dir"] / f"experiment_{timestamp}"
exp_dir_path.mkdir(parents=True, exist_ok=True)
print(f"Experiment tracking directory: {exp_dir_path.resolve()}")

gpu_info = utils.get_gpu_info()
fp16_mode, bf16_mode, compute_dtype, precision_desc = utils.resolve_precision_mode(train_cfg, gpu_info)

gpu_info_snapshot = {
    "cuda_available": gpu_info["cuda_available"],
    "device_count": gpu_info["device_count"],
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "device_name": gpu_info["devices"][0]["name"] if gpu_info["cuda_available"] else "CPU",
    "vram_gb": gpu_info["devices"][0]["total_memory_gb"] if gpu_info["cuda_available"] else "N/A",
    "precision_mode": precision_desc,
    "compute_dtype": str(compute_dtype)
}

with open(exp_dir_path / "gpu_info.json", "w") as f:
    json.dump(gpu_info_snapshot, f, indent=2)

with open(exp_dir_path / "config.yaml", "w") as f:
    yaml.safe_dump(config, f)

print("✓ GPU metadata and config snapshot saved.")

## 6. Load Model and Tokenizer at Creation-Time
We configure 4-bit BitsAndBytes quantization and load the model passing `torch_dtype=compute_dtype` at creation time to prevent `BFloat16` weight leaks.

In [ ]:
from training.scripts import utils

gpu_info = utils.get_gpu_info()
fp16_mode, bf16_mode, compute_dtype, precision_desc = utils.resolve_precision_mode(train_cfg, gpu_info)

try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
    from trl import SFTTrainer
    from datasets import load_dataset
    import bitsandbytes as bnb

    DEPENDENCIES_OK = True
except ImportError as e:
    print(f"Warning: Required packages not fully installed: {e}")
    print("This notebook will execute a mock dry-run.")
    DEPENDENCIES_OK = False

if DEPENDENCIES_OK:
    if gpu_info["cuda_available"]:
        torch.cuda.empty_cache()

    bnb_config = None
    if gpu_info["cuda_available"]:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=compute_dtype
        )

    base_model_name = model_cfg["base_model_name"]

    print(f"Loading tokenizer: {base_model_name}")
    tokenizer = AutoTokenizer.from_pretrained(
        base_model_name,
        trust_remote_code=model_cfg.get("trust_remote_code", True)
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    print(f"Loading base model: {base_model_name}")

    model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        quantization_config=bnb_config,
        device_map=model_cfg.get("device_map", "auto"),
        trust_remote_code=model_cfg.get("trust_remote_code", True),
        torch_dtype=compute_dtype
    )

    # ==========================================================
    # DEBUG: Base model precision inspection
    # ==========================================================
    print("\n========== MODEL PRECISION AUDIT ==========")
    print("Requested compute_dtype :", compute_dtype)
    print("Model config.torch_dtype:", getattr(model.config, "torch_dtype", None))

    dtype_summary = {}
    for _, p in model.named_parameters():
        dtype_summary[str(p.dtype)] = dtype_summary.get(str(p.dtype), 0) + 1

    print("Parameter dtype summary:")
    for k, v in dtype_summary.items():
        print(f"  {k}: {v}")

    print("\nLinear4bit compute dtype:")
    found = False
    for name, module in model.named_modules():
        if isinstance(module, bnb.nn.Linear4bit):
            print("Module:", name)
            print("compute_dtype:", module.compute_dtype)
            found = True
            break

    if not found:
        print("No Linear4bit modules found.")

    print("===========================================\n")

    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.eos_token_id = tokenizer.eos_token_id

    current_embeddings_size = model.get_input_embeddings().num_embeddings
    if len(tokenizer) > current_embeddings_size:
        print(f"Resizing embeddings: {current_embeddings_size} -> {len(tokenizer)}")
        model.resize_token_embeddings(len(tokenizer))
    else:
        print(
            f"Embeddings size ({current_embeddings_size}) sufficient "
            f"for tokenizer length ({len(tokenizer)}). Skipping resize."
        )

    if gpu_info["cuda_available"]:
        model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=True
        )
        model.config.use_cache = False

        print("✓ Model prepared for kbit training.")
        print("✓ Gradient checkpointing enabled.")
        print("✓ KV cache disabled.")

else:
    print("Skipped model loading (dry-run mode).")

## 7. Setup PEFT LoRA Config
Inject Low-Rank Adaptation matrices cleanly with an idempotent `PeftModel` check.

In [ ]:
if DEPENDENCIES_OK:
    peft_config = LoraConfig(
        r=lora_cfg["r"],
        lora_alpha=lora_cfg["lora_alpha"],
        target_modules=lora_cfg["target_modules"],
        lora_dropout=lora_cfg["lora_dropout"],
        bias=lora_cfg["bias"],
        task_type=lora_cfg["task_type"]
    )

    if not isinstance(model, PeftModel):
        model = get_peft_model(model, peft_config)
        print("✓ PEFT LoRA adapter initialized cleanly.")
    else:
        print("✓ Model is already wrapped with PeftModel. Skipping get_peft_model.")

    # 1. LoRA PARAMETER DTYPE BEFORE ENFORCEMENT
    print("\n=== LoRA Parameter Dtypes BEFORE Enforcement ===")
    for name, param in model.named_parameters():
        if param.requires_grad and "lora_" in name:
            print(f"{name}: {param.dtype}")

    # 2. LOADER DTYPE ENFORCEMENT VERIFICATION
    for name, param in model.named_parameters():
        if param.requires_grad and param.dtype != compute_dtype:
            param.data = param.data.to(compute_dtype)

    print("\n=== LoRA Parameter Dtypes AFTER Enforcement ===")
    for name, param in model.named_parameters():
        if param.requires_grad and "lora_" in name:
            print(f"{name}: {param.dtype}")

    print("\nTrainable parameter information:")
    model.print_trainable_parameters()
    
    trainable_dtypes = {p.dtype for p in model.parameters() if p.requires_grad}
    print(f"Trainable parameters dtype(s): {trainable_dtypes}")

    print("\n=== Pre-Training Trainable Parameter Dtype Audit ===")
    for name, p in model.named_parameters():
        if p.requires_grad:
            print(name, p.dtype)

else:
    print("Skipped PEFT setup (dry-run mode).")

In [ ]:
# Instrumentation A: Trainable parameter dtype audit (run after get_peft_model)
if DEPENDENCIES_OK:
    print("\n=== Trainable Parameter Dtype Audit ===")
    bf16_params, fp16_params, other_params = [], [], []
    for name, param in model.named_parameters():
        if param.requires_grad:
            if param.dtype == torch.bfloat16:
                bf16_params.append(name)
            elif param.dtype == torch.float16:
                fp16_params.append(name)
            else:
                other_params.append(f"{name}: {param.dtype}")
    print(f"  BF16 trainable params : {len(bf16_params)} total")
    for n in bf16_params[:5]:
        print(f"    {n}")
    print(f"  FP16 trainable params : {len(fp16_params)} total")
    for n in fp16_params[:5]:
        print(f"    {n}")
    print(f"  Other (FP32/etc) trainable params: {len(other_params)} total")
    for n in other_params[:5]:
        print(f"    {n}")
    if bf16_params:
        raise AssertionError(
            f"ABORT: {len(bf16_params)} BF16 trainable params found. "
            f"First offender: {bf16_params[0]}. "
            "Check compute_dtype — resolve_precision_mode should return "
            "torch.float16 when config fp16=true."
        )
    print("  ✓ No BF16 trainable parameters. Safe to train with FP16 GradScaler.")

In [ ]:
# Instrumentation B: One-shot gradient dtype hook (fires on first backward pass)
if DEPENDENCIES_OK:
    _grad_bf16_reported = {"done": False}

    def _make_grad_dtype_hook(param_name, param_ref):
        def hook(grad):
            if not _grad_bf16_reported["done"] and grad is not None:
                _grad_bf16_reported["done"] = True
                print("\n=== FIRST BACKWARD PASS DIAGNOSTICS ===")
                print(f"Parameter: {param_name}")
                print(f"Parameter dtype: {param_ref.dtype}")
                print(f"Gradient dtype: {grad.dtype}")
                print(f"Device: {param_ref.device}")
                print(f"Requires grad: {param_ref.requires_grad}")
                if grad.dtype == torch.bfloat16:
                    raise RuntimeError(
                        f"BF16 gradient detected on {param_name!r}. "
                        "GradScaler will crash at accelerator.clip_grad_norm_() / unscale_(). "
                        "Verify compute_dtype=torch.float16 and bf16=False in config."
                    )
            return grad
        return hook

    for _pn, _pp in model.named_parameters():
        if _pp.requires_grad:
            _pp.register_hook(_make_grad_dtype_hook(_pn, _pp))
            print(f"  Gradient audit hook registered on: {_pn}")
            break

## 8. Load, Validate, and Format Datasets
Load processed JSONL splits from `PROJECT_ROOT`, validate file integrity, and apply Qwen ChatML template formatting.

In [ ]:
train_jsonl = PROJECT_ROOT / config["dataset"]["train_path"]
val_jsonl = PROJECT_ROOT / config["dataset"]["val_path"]

if DEPENDENCIES_OK:
    from training.scripts import utils
    utils.validate_dataset_schema(train_jsonl)
    utils.validate_dataset_schema(val_jsonl)

    print(f"Loading JSONL splits from: {train_jsonl.parent.resolve()}")
    dataset = load_dataset("json", data_files={
        "train": str(train_jsonl),
        "validation": str(val_jsonl)
    })

    def apply_chat_template(batch):
        formatted_texts = []
        for conv in batch["messages"]:
            if not isinstance(conv, list) or len(conv) == 0:
                raise ValueError("Malformed dataset record: 'messages' must be a non-empty list of message dicts.")
            text = tokenizer.apply_chat_template(conv, tokenize=False)
            formatted_texts.append(text)
        return {"text": formatted_texts}

    formatted_dataset = dataset.map(apply_chat_template, batched=True)
    
    if "text" not in formatted_dataset["train"].column_names:
        raise ValueError("Dataset mapping failed to produce required 'text' column.")

    print("✓ Dataset formatted with Qwen Chat template.")
    print(f"  Training samples: {len(formatted_dataset['train'])}")
    print(f"  Validation samples: {len(formatted_dataset['validation'])}")
else:
    print("Skipped dataset loading (dry-run mode).")

## 9. Dataset Record Inspection
Verify formatted columns and preview sample text.

In [ ]:
if DEPENDENCIES_OK:
    print("--- Dataset Column Names ---")
    print("Train:", formatted_dataset["train"].column_names)
    print("Validation:", formatted_dataset["validation"].column_names)

    print("\n--- Train Record #0 Preview ---")
    print(formatted_dataset["train"][0]["text"])
    print("✓ Dataset pre-flight validation passed.")
else:
    print("Skipped dataset validation (dry-run mode).")

## 10. Training Execution with TrainingArguments & SFTTrainer
Execute training loop using `TrainingArguments` (with programmatically calculated `warmup_steps`) and TRL 1.8.0 `SFTTrainer`.

In [ ]:
if DEPENDENCIES_OK:
    from transformers import TrainingArguments, TrainerCallback
    import torch
    from training.scripts import utils

    # Derive GPU & precision info dynamically
    gpu_info = utils.get_gpu_info()
    fp16_mode, bf16_mode, compute_dtype, precision_desc = utils.resolve_precision_mode(train_cfg, gpu_info)

    # 1. Run pre-flight state validation constraints
    utils.validate_notebook_state(model, tokenizer, peft_config, config, gpu_info, PROJECT_ROOT)
    print("✓ Pre-flight notebook state validation complete. All models, tokens, datasets are fully aligned.")

    # 2. Log run metadata snapshot
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    all_params = sum(p.numel() for p in model.parameters())
    utils.save_run_metadata(
        output_path=exp_dir_path / "run_metadata.json",
        config=config,
        gpu_info=gpu_info,
        precision_desc=precision_desc,
        compute_dtype=str(compute_dtype),
        trainable_params=trainable_params,
        all_params=all_params
    )
    print(f"✓ Run metadata snapshot logged to: {exp_dir_path / 'run_metadata.json'}")

    checkpoint_dir = PROJECT_ROOT / train_cfg["output_dir"] / f"run_{timestamp}"

    # Programmatically compute warmup_steps
    num_train_samples = len(formatted_dataset["train"])
    per_device_batch_size = train_cfg["per_device_train_batch_size"]
    grad_accum_steps = train_cfg["gradient_accumulation_steps"]
    epochs = train_cfg["num_train_epochs"]

    total_steps = max(
        1,
        (num_train_samples // (per_device_batch_size * grad_accum_steps)) * epochs
    )
    computed_warmup_steps = max(
        1,
        int(total_steps * train_cfg.get("warmup_ratio", 0.03))
    )

    print(
        f"Calculated training steps: "
        f"Total Steps = {total_steps} | Warmup Steps = {computed_warmup_steps}"
    )

    # Resume from checkpoint integrity verification
    resume_checkpoint = None
    if train_cfg.get("resume_from_checkpoint", False):
        checkpoint_root = PROJECT_ROOT / train_cfg["output_dir"]
        runs = [d for d in checkpoint_root.iterdir() if d.is_dir() and d.name.startswith("run_")]
        if runs:
            runs.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_run = runs[0]
            checkpoints = [d for d in latest_run.iterdir() if d.is_dir() and d.name.startswith("checkpoint-")]
            if checkpoints:
                checkpoints.sort(key=lambda x: int(x.name.split("-")[-1]), reverse=True)
                latest_checkpoint_dir = checkpoints[0]
                if utils.validate_checkpoint_integrity(latest_checkpoint_dir):
                    print(f"Checkpoint integrity verified. Resuming training from: {latest_checkpoint_dir}")
                    resume_checkpoint = str(latest_checkpoint_dir)
                else:
                    print(
                        f"WARNING: Checkpoint integrity check failed for '{latest_checkpoint_dir}'. "
                        "Required files are missing or incomplete. Starting a fresh training run instead."
                    )

    # Use eval_strategy instead of deprecated evaluation_strategy (Transformers 5.x compatibility)
    training_args = TrainingArguments(
        output_dir=str(checkpoint_dir),
        num_train_epochs=train_cfg["num_train_epochs"],
        per_device_train_batch_size=train_cfg["per_device_train_batch_size"],
        per_device_eval_batch_size=train_cfg["per_device_eval_batch_size"],
        gradient_accumulation_steps=train_cfg["gradient_accumulation_steps"],
        learning_rate=float(train_cfg["learning_rate"]),
        weight_decay=train_cfg["weight_decay"],
        warmup_steps=computed_warmup_steps,
        lr_scheduler_type=train_cfg["lr_scheduler_type"],
        logging_steps=train_cfg["logging_steps"],
        save_strategy=train_cfg["save_strategy"],
        eval_strategy=train_cfg.get("eval_strategy", "epoch"),
        fp16=fp16_mode,
        bf16=bf16_mode,
        report_to="none",
        seed=config["dataset"].get("random_seed", 42),
        logging_first_step=True,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False
    )

    class MetricsHistory(TrainerCallback):
        def __init__(self, target_dir):
            self.target_dir = target_dir
            self.log_history = []
        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs:
                logs["step"] = state.global_step
                logs["epoch"] = state.epoch
                self.log_history.append(logs)
                with open(self.target_dir / "metrics_history.json", "w") as hf:
                    json.dump(self.log_history, hf, indent=2)

    history_callback = MetricsHistory(exp_dir_path)

    sft_kwargs = utils.get_sft_trainer_kwargs(tokenizer)
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=formatted_dataset["train"],
        eval_dataset=formatted_dataset["validation"],
        callbacks=[history_callback],
        **sft_kwargs
    )

    # 3. ACCELERATE MIXED PRECISION STATE
    print("\n=== Accelerate State ===")
    if hasattr(trainer, "accelerator") and hasattr(trainer.accelerator, "state"):
        print("Mixed precision:", trainer.accelerator.state.mixed_precision)
    else:
        print("Mixed precision: (accelerator state not initialized in dry-run)")

    print("TrainingArguments.fp16:", training_args.fp16)
    print("TrainingArguments.bf16:", training_args.bf16)
    print("compute_dtype:", compute_dtype)

    # 4. AUTOCAST STATE
    print("\n=== Autocast Diagnostics ===")
    print("torch autocast enabled:", torch.is_autocast_enabled())

    if torch.cuda.is_available():
        try:
            print("CUDA autocast dtype:", torch.get_autocast_gpu_dtype())
        except Exception as e:
            print("Unable to query autocast dtype:", e)

    print("\nStarting training execution loop...")
    train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)

    metrics = train_result.metrics
    eval_metrics = trainer.evaluate()
    metrics.update(eval_metrics)

    trainer.model.save_pretrained(str(exp_dir_path / "adapter"))
    tokenizer.save_pretrained(str(exp_dir_path / "adapter"))

    trainer.model.save_pretrained(str(checkpoint_dir))
    tokenizer.save_pretrained(str(checkpoint_dir))

    with open(exp_dir_path / "metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    print("✓ Training completed successfully!")
    print(f"Metrics: {metrics}")
    print(f"Adapter saved to: {(exp_dir_path / 'adapter').resolve()}")
    print(f"Checkpoint saved to: {checkpoint_dir.resolve()}")

else:
    print("Executing simulated dry-run loop...")
    dummy_metrics = {
        "train_runtime": 120.5,
        "train_samples_per_second": 8.2,
        "train_loss": 0.1245,
        "eval_loss": 0.1412,
        "epoch": 3.0
    }
    with open(exp_dir_path / "metrics.json", "w") as f:
        json.dump(dummy_metrics, f, indent=2)

    (exp_dir_path / "adapter").mkdir(parents=True, exist_ok=True)
    with open(exp_dir_path / "adapter/adapter_config.json", "w") as f:
        f.write('{"peft_type":"LORA","r":16,"lora_alpha":32}')

    mock_checkpoint = PROJECT_ROOT / train_cfg["output_dir"] / f"run_{timestamp}"
    mock_checkpoint.mkdir(parents=True, exist_ok=True)
    with open(mock_checkpoint / "adapter_config.json", "w") as f:
        f.write('{"peft_type":"LORA","r":16,"lora_alpha":32}')
    print("✓ Dry-run completed.")